In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)

In [2]:
from pathlib import Path

BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data" / "raw"

print("Project:", BASE_DIR)
print("Data:", DATA_DIR)

for file in DATA_DIR.iterdir():
    print(file.name)

Project: C:\Users\Mitalika\OneDrive\Desktop\enterprise_hr_ai
Data: C:\Users\Mitalika\OneDrive\Desktop\enterprise_hr_ai\data\raw
Cleaned_HR_Data_Analysis.csv
Employee_Performance_Dataset.csv
employee_performance_pro.csv
essential_skills.csv
Messy_HR_Dataset_Detailed.csv
Occupation Data.xlsx
Software Skills.xlsx
WA_Fn-UseC_-HR-Employee-Attrition.csv


In [3]:
datasets = {}

for file in DATA_DIR.iterdir():
    if file.suffix.lower() == ".csv":
        datasets[file.stem] = pd.read_csv(file)
    elif file.suffix.lower() in [".xlsx", ".xls"]:
        datasets[file.stem] = pd.read_excel(file)

print("Datasets loaded:", len(datasets))

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

Datasets loaded: 8
Cleaned_HR_Data_Analysis: (2845, 28)
Employee_Performance_Dataset: (5000, 13)
employee_performance_pro: (500, 24)
essential_skills: (18200, 15)
Messy_HR_Dataset_Detailed: (3150, 39)
Occupation Data: (1016, 3)
Software Skills: (31821, 7)
WA_Fn-UseC_-HR-Employee-Attrition: (1470, 35)


In [4]:
audit_summary = []

for name, df in datasets.items():
    audit_summary.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicate_rows": df.duplicated().sum(),
        "missing_cells": df.isna().sum().sum(),
        "missing_percentage": round(
            (df.isna().sum().sum() / df.size) * 100, 2
        )
    })

audit_df = pd.DataFrame(audit_summary)

audit_df.sort_values("rows", ascending=False)

,dataset,rows,columns,duplicate_rows,missing_cells,missing_percentage
6,Software Skills,31821,7,0,0,0.00
3,essential_skills,18200,15,0,9100,3.33
1,Employee_Performance_Dataset,5000,13,0,0,0.00
4,Messy_HR_Dataset_Detailed,3150,39,150,3088,2.51
0,Cleaned_HR_Data_Analysis,2845,28,0,0,0.00
7,WA_Fn-UseC_-HR-Employee-Attrition,1470,35,0,0,0.00
5,Occupation Data,1016,3,0,0,0.00
2,employee_performance_pro,500,24,0,319,2.66


In [6]:
for name, df in datasets.items():
    print("\n" + "=" * 100)
    print(f"DATASET: {name}")
    print(f"SHAPE: {df.shape}")
    print("=" * 100)

    for col in df.columns:
        print(
            f"{col:40} | "
            f"dtype={str(df[col].dtype):10} | "
            f"unique={df[col].nunique(dropna=False)}"
        )


DATASET: Cleaned_HR_Data_Analysis
SHAPE: (2845, 28)
Employee ID                              | dtype=int64      | unique=2845
StartDate                                | dtype=object     | unique=1472
Title                                    | dtype=object     | unique=32
BusinessUnit                             | dtype=object     | unique=10
EmployeeStatus                           | dtype=object     | unique=2
EmployeeType                             | dtype=object     | unique=3
PayZone                                  | dtype=object     | unique=3
EmployeeClassificationType               | dtype=object     | unique=3
DepartmentType                           | dtype=object     | unique=6
Division                                 | dtype=object     | unique=25
DOB                                      | dtype=object     | unique=2664
State                                    | dtype=object     | unique=28
GenderCode                               | dtype=object     | unique=2
RaceDesc   

In [8]:
for name, df in datasets.items():

    print("\n" + "=" * 100)
    print(f"DATASET: {name}")
    print("=" * 100)

    missing = df.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)

    if len(missing) == 0:
        print("No missing values.")
    else:
        missing_df = pd.DataFrame({
            "missing_count": missing,
            "missing_percentage": (
                missing / len(df) * 100
            ).round(2)
        })

        print(missing_df)


DATASET: Cleaned_HR_Data_Analysis
No missing values.

DATASET: Employee_Performance_Dataset
No missing values.

DATASET: employee_performance_pro
                      missing_count  missing_percentage
CustomerSatisfaction            319                63.8

DATASET: essential_skills
              missing_count  missing_percentage
Not Relevant           9100                50.0

DATASET: Messy_HR_Dataset_Detailed
                        missing_count  missing_percentage
ExitDate                         1544               49.02
TerminationDescription           1544               49.02

DATASET: Occupation Data
No missing values.

DATASET: Software Skills
No missing values.

DATASET: WA_Fn-UseC_-HR-Employee-Attrition
No missing values.


In [9]:
messy = datasets["Messy_HR_Dataset_Detailed"]

print("Total rows:", len(messy))
print("Unique Employee IDs:", messy["Employee ID"].nunique())
print("Duplicate Employee ID rows:", messy["Employee ID"].duplicated().sum())

Total rows: 3150
Unique Employee IDs: 3000
Duplicate Employee ID rows: 150


In [10]:
duplicate_ids = messy.loc[
    messy["Employee ID"].duplicated(keep=False),
    "Employee ID"
].value_counts()

print("Number of Employee IDs appearing more than once:",
      len(duplicate_ids))

print("\nMost frequent duplicate Employee IDs:")
print(duplicate_ids.head(20))

Number of Employee IDs appearing more than once: 150

Most frequent duplicate Employee IDs:
Employee ID
3427    2
3441    2
3471    2
3472    2
3478    2
3479    2
3490    2
3520    2
3529    2
3571    2
3579    2
3601    2
3621    2
3623    2
3630    2
3678    2
3693    2
3697    2
3716    2
3718    2
Name: count, dtype: int64


In [12]:
duplicate_sample = messy[
    messy["Employee ID"].isin(duplicate_ids.head(5).index)
].sort_values("Employee ID")

display(duplicate_sample)

,Unnamed: 0,FirstName,LastName,StartDate,ExitDate,Title,Supervisor,ADEmail,BusinessUnit,EmployeeStatus,EmployeeType,PayZone,EmployeeClassificationType,TerminationType,TerminationDescription,DepartmentType,Division,DOB,State,JobFunctionDescription,GenderCode,LocationCode,RaceDesc,MaritalDesc,Performance Score,Current Employee Rating,Employee ID,Survey Date,Engagement Score,Satisfaction Score,Work-Life Balance Score,Training Date,Training Program Name,Training Type,Training Outcome,Location,Trainer,Training Duration(Days),Training Cost
0,0,Uriah,Bridges,20-Sep-19,NaN,Production Technician I,Peter Oneill,uriah.bridges@bilearner.com,CCDR,Active,Contract,Zone C,Temporary,Unk,NaN,Production,Finance & Accounting,07-10-1969,MA,Accounting,Female,34904,White,Widowed,Fully Meets,4,3427,14-01-2023,1,2,3,15-Jul-23,Leadership Development,Internal,Failed,South Marisa,Taylor Rodriguez,2,606.11
3116,0,Uriah,Bridges,20-Sep-19,NaN,Production Technician I,Peter Oneill,uriah.bridges@bilearner.com,CCDR,Active,Contract,Zone C,Temporary,Unk,NaN,Production,Finance & Accounting,07-10-1969,MA,Accounting,Female,34904,White,Widowed,Fully Meets,4,3427,14-01-2023,1,2,3,15-Jul-23,Leadership Development,Internal,Failed,South Marisa,Taylor Rodriguez,2,606.11
3047,14,Prater,Jeremy,28-Apr-19,NaN,Area Sales Manager,Tyler Lewis,prater.jeremy@bilearner.com,BPC,Active,Part-Time,Zone A,Part-Time,Unk,NaN,Sales,General - Con,21-11-1989,NV,Lineman,Male,89139,Asian,Widowed,Exceeds,4,3441,30-10-2022,4,2,5,08-Jan-23,Project Management,Internal,Passed,Carrollside,Robert Kane,4,450.20
14,14,Prater,Jeremy,28-Apr-19,NaN,Area Sales Manager,Tyler Lewis,prater.jeremy@bilearner.com,BPC,Active,Part-Time,Zone A,Part-Time,Unk,NaN,Sales,General - Con,21-11-1989,NV,Lineman,Male,89139,Asian,Widowed,Exceeds,4,3441,30-10-2022,4,2,5,08-Jan-23,Project Management,Internal,Passed,Carrollside,Robert Kane,4,450.20
3020,44,Jonathan,Adkins,29-Feb-20,NaN,Area Sales Manager,John Marshall,jonathan.adkins@bilearner.com,CCDR,Active,Part-Time,Zone C,Temporary,Unk,NaN,Sales,Field Operations,03-01-1967,TX,Foreman,Male,64350,Asian,Married,Fully Meets,4,3471,30-04-2023,2,4,1,23-Mar-23,Customer Service,External,Passed,West Jonathan,Frank Joyce,3,109.98
44,44,Jonathan,Adkins,29-Feb-20,NaN,Area Sales Manager,John Marshall,jonathan.adkins@bilearner.com,CCDR,Active,Part-Time,Zone C,Temporary,Unk,NaN,Sales,Field Operations,03-01-1967,TX,Foreman,Male,64350,Asian,Married,Fully Meets,4,3471,30-04-2023,2,4,1,23-Mar-23,Customer Service,External,Passed,West Jonathan,Frank Joyce,3,109.98
3101,45,Nevaeh,Soto,15-Jan-23,NaN,Area Sales Manager,Jessica Chang,nevaeh.soto@bilearner.com,SVG,Active,Contract,Zone C,Full-Time,Unk,NaN,Sales,Project Management - Con,12-01-1982,TX,Director,Male,74124,Other,Married,Exceeds,4,3472,26-07-2023,5,5,4,12-Feb-23,Technical Skills,External,Completed,Port Karatown,Matthew Meyer,5,537.24
45,45,Nevaeh,Soto,15-Jan-23,NaN,Area Sales Manager,Jessica Chang,nevaeh.soto@bilearner.com,SVG,Active,Contract,Zone C,Full-Time,Unk,NaN,Sales,Project Management - Con,12-01-1982,TX,Director,Male,74124,Other,Married,Exceeds,4,3472,26-07-2023,5,5,4,12-Feb-23,Technical Skills,External,Completed,Port Karatown,Matthew Meyer,5,537.24
3064,51,Thomas,Chandler,17-Sep-18,09-Jan-19,Area Sales Manager,Richard Hodges,thomas.chandler@bilearner.com,WBL,Active,Full-Time,Zone C,Temporary,Resignation,Read family dark scene scene guess.,Sales,Wireline Construction,10-10-1957,KY,Groundman,Female,45149,Hispanic,Married,Fully Meets,2,3478,29-11-2022,3,4,1,05-Oct-22,Technical Skills,External,Incomplete,Port Matthew,Daniel Solomon,1,526.74
51,51,Thomas,Chandler,17-Sep-18,09-Jan-19,Area Sales Manager,Richard Hodges,thomas.chandler@bilearner.com,WBL,Active,Full-Time,Zone C,Temporary,Resignation,Read family dark scene scene guess.,Sales,Wireline Construction,10-10-1957,KY,Groundman,Female,45149,Hispanic,Married,Fully Meets,2,3478,29-11-2022,3,4,1,05-Oct-22,Technical Skills,External,Incomplete,Port Matthew,Daniel Solomon,1,526.74


In [13]:
messy = datasets["Messy_HR_Dataset_Detailed"]

# Count exact duplicate rows
exact_duplicates = messy.duplicated(keep=False)

print("Rows involved in exact duplicates:", exact_duplicates.sum())
print("Number of unique exact duplicate records:",
      messy[exact_duplicates].drop_duplicates().shape[0])

print("\nTotal duplicated rows according to pandas:",
      messy.duplicated().sum())

Rows involved in exact duplicates: 300
Number of unique exact duplicate records: 150

Total duplicated rows according to pandas: 150


In [14]:
messy = datasets["Messy_HR_Dataset_Detailed"]

# 1. Active employees who have an ExitDate
active_with_exit = messy[
    (messy["EmployeeStatus"] == "Active") &
    (messy["ExitDate"].notna())
]

print("Active employees with ExitDate:",
      len(active_with_exit))


# 2. Active employees with a termination type other than 'Unk'
active_with_termination = messy[
    (messy["EmployeeStatus"] == "Active") &
    (messy["TerminationType"].notna()) &
    (messy["TerminationType"] != "Unk")
]

print("Active employees with termination type:",
      len(active_with_termination))


# 3. Inactive employees without an ExitDate
inactive_without_exit = messy[
    (messy["EmployeeStatus"] != "Active") &
    (messy["ExitDate"].isna())
]

print("Non-active employees without ExitDate:",
      len(inactive_without_exit))


# 4. Show EmployeeStatus distribution
print("\nEmployee Status:")
print(messy["EmployeeStatus"].value_counts(dropna=False))


# 5. Show TerminationType distribution
print("\nTermination Type:")
print(messy["TerminationType"].value_counts(dropna=False))

Active employees with ExitDate: 1039
Active employees with termination type: 1039
Non-active employees without ExitDate: 0

Employee Status:
EmployeeStatus
Active                    2583
Voluntarily Terminated     339
Leave of Absence            89
Future Start                70
Terminated for Cause        69
Name: count, dtype: int64

Termination Type:
TerminationType
Unk            1544
Voluntary       409
Involuntary     407
Resignation     397
Retirement      393
Name: count, dtype: int64


In [15]:
# Cross-tabulation between EmployeeStatus and TerminationType

status_termination = pd.crosstab(
    messy["EmployeeStatus"],
    messy["TerminationType"],
    margins=True
)

display(status_termination)

TerminationType,Involuntary,Resignation,Retirement,Unk,Voluntary,All
EmployeeStatus,,,,,,
Active,254,264,255,1544,266,2583
Future Start,19,11,23,0,17,70
Leave of Absence,20,23,24,0,22,89
Terminated for Cause,23,22,10,0,14,69
Voluntarily Terminated,91,77,81,0,90,339
All,407,397,393,1544,409,3150


In [16]:
exit_analysis = messy.groupby("EmployeeStatus").agg(
    total_records=("Employee ID", "count"),
    exit_dates_present=("ExitDate", lambda x: x.notna().sum()),
    exit_dates_missing=("ExitDate", lambda x: x.isna().sum())
)

exit_analysis["exit_date_percentage"] = (
    exit_analysis["exit_dates_present"]
    / exit_analysis["total_records"]
    * 100
).round(2)

display(exit_analysis)

,total_records,exit_dates_present,exit_dates_missing,exit_date_percentage
EmployeeStatus,,,,
Active,2583,1039,1544,40.22
Future Start,70,70,0,100.00
Leave of Absence,89,89,0,100.00
Terminated for Cause,69,69,0,100.00
Voluntarily Terminated,339,339,0,100.00


In [17]:
termination_analysis = messy.groupby("EmployeeStatus").agg(
    total_records=("Employee ID", "count"),
    termination_type_unknown=(
        "TerminationType",
        lambda x: (x == "Unk").sum()
    ),
    termination_type_known=(
        "TerminationType",
        lambda x: (x != "Unk").sum()
    )
)

display(termination_analysis)

,total_records,termination_type_unknown,termination_type_known
EmployeeStatus,,,
Active,2583,1544,1039
Future Start,70,0,70
Leave of Absence,89,0,89
Terminated for Cause,69,0,69
Voluntarily Terminated,339,0,339


In [ ]:
# 2 STARTS

In [18]:
# Compare important categorical fields across HR datasets

hr_datasets = [
    "Cleaned_HR_Data_Analysis",
    "Employee_Performance_Dataset",
    "employee_performance_pro",
    "Messy_HR_Dataset_Detailed",
    "WA_Fn-UseC_-HR-Employee-Attrition"
]

for name in hr_datasets:
    df = datasets[name]

    print("\n" + "=" * 100)
    print(name)
    print("=" * 100)

    for col in ["Department", "JobRole", "Job Role", "Title"]:
        if col in df.columns:
            print(f"\n{col} — {df[col].nunique()} unique values")
            print(df[col].dropna().unique()[:30])


Cleaned_HR_Data_Analysis

Title — 32 unique values
['Production Technician I' 'Area Sales Manager' 'Production Technician II'
 'IT Support' 'Network Engineer' 'Sr. Network Engineer'
 'Principal Data Architect' 'Enterprise Architect'
 'Database Administrator' 'Data Analyst' 'Sr. DBA' 'Data Analyst '
 'Data Architect' 'CIO' 'BI Director' 'Sr. Accountant'
 'Software Engineering Manager' 'Software Engineer'
 'Shared Services Manager' 'Senior BI Developer' 'Production Manager'
 'President & CEO' 'Administrative Assistant' 'Accountant I'
 'BI Developer' 'Sales Manager' 'IT Manager - Support'
 'IT Manager - Infra' 'IT Manager - DB' 'Director of Sales']

Employee_Performance_Dataset

Department — 5 unique values
['Sales' 'Marketing' 'Finance' 'HR' 'IT']

Job Role — 15 unique values
['Sales Executive' 'Marketing Executive' 'Accountant' 'Content Strategist'
 'Employee Relations' 'Business Development' 'Cybersecurity Specialist'
 'Account Manager' 'HR Manager' 'Auditor' 'Data Analyst'
 'Recruitm

In [19]:
keywords = [
    "skill",
    "technology",
    "software",
    "certification",
    "competenc",
    "qualification",
    "education",
    "training"
]

for name, df in datasets.items():

    print("\n" + "=" * 100)
    print(name)
    print("=" * 100)

    for col in df.columns:
        if any(keyword in col.lower() for keyword in keywords):
            print(col)


Cleaned_HR_Data_Analysis
Training Date
Training Program Name
Training Type
Training Outcome
Training Duration(Days)
Training Cost

Employee_Performance_Dataset
Training Hours

employee_performance_pro
EducationLevel
TrainingHours

essential_skills

Messy_HR_Dataset_Detailed
Training Date
Training Program Name
Training Type
Training Outcome
Training Duration(Days)
Training Cost

Occupation Data

Software Skills
Hot Technology

WA_Fn-UseC_-HR-Employee-Attrition
Education
EducationField
TrainingTimesLastYear


In [20]:
for name in ["Occupation Data", "essential_skills", "Software Skills"]:
    
    df = datasets[name]
    
    print("\n" + "=" * 100)
    print(name)
    print("=" * 100)
    
    print("Columns:")
    print(df.columns.tolist())
    
    print("\nFirst 5 rows:")
    display(df.head())


Occupation Data
Columns:
['O*NET-SOC Code', 'Title', 'Description']

First 5 rows:


,O*NET-SOC Code,Title,Description
0,11-1011.00,Chief Executives,Determine and formulate policies and provide o...
1,11-1011.03,Chief Sustainability Officers,"Communicate and coordinate with management, sh..."
2,11-1021.00,General and Operations Managers,"Plan, direct, or coordinate the operations of ..."
3,11-1031.00,Legislators,"Develop, introduce, or enact laws and statutes..."
4,11-2011.00,Advertising and Promotions Managers,"Plan, direct, or coordinate advertising polici..."



essential_skills
Columns:
['O*NET-SOC Code', 'Title', 'Element ID', 'Element Name', 'Scale ID', 'Scale Name', 'Data Value', 'N', 'Standard Error', 'Lower CI Bound', 'Upper CI Bound', 'Recommend Suppress', 'Not Relevant', 'Date', 'Domain Source']

First 5 rows:


,O*NET-SOC Code,Title,Element ID,Element Name,Scale ID,Scale Name,Data Value,N,Standard Error,Lower CI Bound,Upper CI Bound,Recommend Suppress,Not Relevant,Date,Domain Source
0,11-1011.00,Chief Executives,2.A.1.a,Reading Comprehension,IM,Importance,4.12,8,0.1250,3.8800,4.3700,N,NaN,08/2023,Analyst
1,11-1011.00,Chief Executives,2.A.1.a,Reading Comprehension,LV,Level,4.62,8,0.1830,4.2664,4.9836,N,N,08/2023,Analyst
2,11-1011.00,Chief Executives,2.A.1.b,Active Listening,IM,Importance,4.00,8,0.0000,4.0000,4.0000,N,NaN,08/2023,Analyst
3,11-1011.00,Chief Executives,2.A.1.b,Active Listening,LV,Level,4.75,8,0.1637,4.4292,5.0708,N,N,08/2023,Analyst
4,11-1011.00,Chief Executives,2.A.1.c,Writing,IM,Importance,4.12,8,0.1250,3.8800,4.3700,N,NaN,08/2023,Analyst



Software Skills
Columns:
['O*NET-SOC Code', 'Title', 'Workplace Example', 'Element ID', 'Element Name', 'Hot Technology', 'In Demand']

First 5 rows:


,O*NET-SOC Code,Title,Workplace Example,Element ID,Element Name,Hot Technology,In Demand
0,11-1011.00,Chief Executives,Adobe Acrobat,2.E.5.b,Document management software,Y,N
1,11-1011.00,Chief Executives,AdSense Tracker,2.E.6.f,Data base user interface and query software,N,N
2,11-1011.00,Chief Executives,Atlassian JIRA,2.E.5.a,Content workflow software,Y,N
3,11-1011.00,Chief Executives,Blackbaud The Raiser's Edge,2.E.6.c,Customer relationship management CRM software,N,N
4,11-1011.00,Chief Executives,ComputerEase construction accounting software,2.E.2.a,Accounting software,N,N


In [21]:
# Get O*NET occupation titles
onet = datasets["Occupation Data"]

# Collect employee job titles from our HR datasets
employee_titles = set()

for name in [
    "Cleaned_HR_Data_Analysis",
    "Employee_Performance_Dataset",
    "employee_performance_pro",
    "Messy_HR_Dataset_Detailed",
    "WA_Fn-UseC_-HR-Employee-Attrition"
]:
    
    df = datasets[name]
    
    for col in ["Title", "Job Role", "JobRole"]:
        if col in df.columns:
            employee_titles.update(
                df[col]
                .dropna()
                .astype(str)
                .str.strip()
                .unique()
            )

print("Unique employee job titles:", len(employee_titles))

print("\nEmployee job titles:")
for title in sorted(employee_titles):
    print("-", title)

print("\nTotal O*NET occupations:", len(onet))

Unique employee job titles: 60

Employee job titles:
- Account Manager
- Accountant
- Accountant I
- Administrative Assistant
- Area Sales Manager
- Auditor
- BI Developer
- BI Director
- Business Development
- CIO
- Content Lead
- Content Strategist
- Cybersecurity Specialist
- Data Analyst
- Data Architect
- Database Administrator
- Developer
- Director of Operations
- Director of Sales
- Employee Relations
- Engineer
- Enterprise Architect
- Financial Analyst
- HR Executive
- HR Manager
- Healthcare Representative
- Helpdesk
- Human Resources
- IT Director
- IT Manager - DB
- IT Manager - Infra
- IT Manager - Support
- IT Support
- Laboratory Technician
- Manager
- Manufacturing Director
- Marketing Executive
- Network Engineer
- President & CEO
- Principal Data Architect
- Production Manager
- Production Technician I
- Production Technician II
- Recruitment Specialist
- Research Director
- Research Scientist
- SEO Analyst
- SEO Specialist
- Sales Executive
- Sales Manager
- Sales R

In [22]:
hr_datasets = [
    "Cleaned_HR_Data_Analysis",
    "Employee_Performance_Dataset",
    "employee_performance_pro",
    "Messy_HR_Dataset_Detailed",
    "WA_Fn-UseC_-HR-Employee-Attrition"
]

for name in hr_datasets:

    df = datasets[name]

    print("\n" + "=" * 100)
    print(name)
    print("=" * 100)

    print("Rows:", df.shape[0])
    print("Columns:", df.shape[1])

    print("\nColumn names:")
    for i, col in enumerate(df.columns, 1):
        print(f"{i:2}. {col}")


Cleaned_HR_Data_Analysis
Rows: 2845
Columns: 28

Column names:
 1. Employee ID
 2. StartDate
 3. Title
 4. BusinessUnit
 5. EmployeeStatus
 6. EmployeeType
 7. PayZone
 8. EmployeeClassificationType
 9. DepartmentType
10. Division
11. DOB
12. State
13. GenderCode
14. RaceDesc
15. MaritalDesc
16. Performance Score
17. Current Employee Rating
18. Survey Date
19. Engagement Score
20. Satisfaction Score
21. Work-Life Balance Score
22. Training Date
23. Training Program Name
24. Training Type
25. Training Outcome
26. Training Duration(Days)
27. Training Cost
28. Age

Employee_Performance_Dataset
Rows: 5000
Columns: 13

Column names:
 1. Employee ID
 2. Name
 3. Department
 4. Job Role
 5. Performance Score
 6. KPI Score
 7. Attendance (%)
 8. Peer Rating
 9. Task Completion (%)
10. Work Hours Logged
11. Manager Feedback
12. Training Hours
13. Promotion Eligibility

employee_performance_pro
Rows: 500
Columns: 24

Column names:
 1. EmployeeID
 2. Name
 3. Gender
 4. Age
 5. Department
 6. Jo

In [23]:
df = datasets["Cleaned_HR_Data_Analysis"]

print("=" * 100)
print("SHAPE")
print("=" * 100)
print(df.shape)

print("\n" + "=" * 100)
print("DATA TYPES")
print("=" * 100)
print(df.dtypes)

print("\n" + "=" * 100)
print("UNIQUE VALUES")
print("=" * 100)

for col in df.columns:
    print(f"{col}: {df[col].nunique()}")

print("\n" + "=" * 100)
print("NUMERIC SUMMARY")
print("=" * 100)

display(df.describe(include="all").T)

SHAPE
(2845, 28)

DATA TYPES
Employee ID                     int64
StartDate                      object
Title                          object
BusinessUnit                   object
EmployeeStatus                 object
EmployeeType                   object
PayZone                        object
EmployeeClassificationType     object
DepartmentType                 object
Division                       object
DOB                            object
State                          object
GenderCode                     object
RaceDesc                       object
MaritalDesc                    object
Performance Score              object
Current Employee Rating         int64
Survey Date                    object
Engagement Score                int64
Satisfaction Score              int64
Work-Life Balance Score         int64
Training Date                  object
Training Program Name          object
Training Type                  object
Training Outcome               object
Training Duration(Day

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Employee ID,2845.0,NaN,NaN,NaN,2470.591916,859.450107,1001.0,1736.0,2456.0,3197.0,4000.0
StartDate,2845,1472,04-Mar-22,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Title,2845,32,Production Technician I,1241,NaN,NaN,NaN,NaN,NaN,NaN,NaN
BusinessUnit,2845,10,NEL,291,NaN,NaN,NaN,NaN,NaN,NaN,NaN
EmployeeStatus,2845,2,Active,2458,NaN,NaN,NaN,NaN,NaN,NaN,NaN
EmployeeType,2845,3,Full-Time,997,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PayZone,2845,3,Zone A,1013,NaN,NaN,NaN,NaN,NaN,NaN,NaN
EmployeeClassificationType,2845,3,Temporary,980,NaN,NaN,NaN,NaN,NaN,NaN,NaN
DepartmentType,2845,6,Production,1910,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Division,2845,25,Field Operations,747,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
df = datasets["Cleaned_HR_Data_Analysis"]

# Columns where we need to understand the actual categories
categorical_columns = [
    "EmployeeStatus",
    "Performance Score",
    "Current Employee Rating",
    "Engagement Score",
    "Satisfaction Score",
    "Work-Life Balance Score",
    "Training Program Name",
    "Training Type",
    "Training Outcome",
    "EmployeeType",
    "EmployeeClassificationType",
    "DepartmentType"
]

for col in categorical_columns:
    print("\n" + "=" * 80)
    print(col)
    print("=" * 80)
    print(df[col].value_counts(dropna=False))


EmployeeStatus
EmployeeStatus
Active        2458
Terminated     387
Name: count, dtype: int64

Performance Score
Performance Score
Fully Meets          2251
Exceeds               346
Needs Improvement     162
PIP                    86
Name: count, dtype: int64

Current Employee Rating
Current Employee Rating
3    1451
2     483
4     403
5     256
1     252
Name: count, dtype: int64

Engagement Score
Engagement Score
1    628
2    576
5    557
4    552
3    532
Name: count, dtype: int64

Satisfaction Score
Satisfaction Score
4    612
3    569
5    565
1    562
2    537
Name: count, dtype: int64

Work-Life Balance Score
Work-Life Balance Score
3    599
1    575
5    558
4    558
2    555
Name: count, dtype: int64

Training Program Name
Training Program Name
Communication Skills      633
Project Management        585
Leadership Development    544
Technical Skills          543
Customer Service          540
Name: count, dtype: int64

Training Type
Training Type
External    1424
Internal  

In [25]:
print("\n" + "=" * 80)
print("AGE DISTRIBUTION")
print("=" * 80)

print(df["Age"].describe())

print("\nEmployees younger than 18:")
display(
    df[df["Age"] < 18][
        ["Employee ID", "DOB", "StartDate", "Age", "EmployeeType", "EmployeeStatus", "Title"]
    ]
)

print("\nEmployees older than 65:")
display(
    df[df["Age"] > 65][
        ["Employee ID", "DOB", "StartDate", "Age", "EmployeeType", "EmployeeStatus", "Title"]
    ]
)


AGE DISTRIBUTION
count    2845.000000
mean       49.448506
std        17.689179
min        17.000000
25%        34.000000
50%        49.000000
75%        65.000000
max        82.000000
Name: Age, dtype: float64

Employees younger than 18:


,Employee ID,DOB,StartDate,Age,EmployeeType,EmployeeStatus,Title
1228,1743,03-06-2001,30-Nov-18,17,Full-Time,Active,Production Technician I
1518,2038,04-05-2001,02-Sep-18,17,Part-Time,Active,Production Technician I



Employees older than 65:


,Employee ID,DOB,StartDate,Age,EmployeeType,EmployeeStatus,Title
5,3432,03-04-1949,17-Jan-20,71,Contract,Active,Area Sales Manager
6,3433,01-07-1942,06-Apr-22,80,Full-Time,Active,Area Sales Manager
9,3436,11-11-1949,21-Jan-22,73,Part-Time,Active,Area Sales Manager
11,3438,06-04-1948,10-Aug-18,70,Full-Time,Active,Area Sales Manager
13,3440,06-11-1951,05-Dec-19,68,Contract,Active,Area Sales Manager
...,...,...,...,...,...,...,...
2826,3407,26-02-1953,15-Apr-21,68,Part-Time,Active,Production Technician II
2828,3409,15-03-1943,08-Oct-22,79,Contract,Active,Production Technician II
2830,3411,20-03-1949,06-Oct-22,73,Full-Time,Active,Production Technician II
2838,3419,05-12-1952,14-Oct-19,67,Contract,Active,Production Technician I


In [26]:
df = datasets["Cleaned_HR_Data_Analysis"].copy()

date_columns = [
    "StartDate",
    "DOB",
    "Survey Date",
    "Training Date"
]

for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce")

print("=" * 100)
print("DATE RANGES")
print("=" * 100)

for col in date_columns:
    print(f"\n{col}")
    print("Min:", df[col].min())
    print("Max:", df[col].max())

print("\n" + "=" * 100)
print("DATE NULLS AFTER CONVERSION")
print("=" * 100)

print(df[date_columns].isna().sum())

C:\Users\Mitalika\AppData\Local\Temp\ipykernel_18064\97709031.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors="coerce")


DATE RANGES

StartDate
Min: 2018-08-07 00:00:00
Max: 2023-08-06 00:00:00

DOB
Min: 1941-02-10 00:00:00
Max: 2001-11-04 00:00:00

Survey Date
Min: 2022-08-05 00:00:00
Max: 2023-08-05 00:00:00

Training Date
Min: 2022-08-05 00:00:00
Max: 2023-08-05 00:00:00

DATE NULLS AFTER CONVERSION
StartDate           0
DOB              1727
Survey Date         0
Training Date       0
dtype: int64


C:\Users\Mitalika\AppData\Local\Temp\ipykernel_18064\97709031.py:11: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df[col] = pd.to_datetime(df[col], errors="coerce")
C:\Users\Mitalika\AppData\Local\Temp\ipykernel_18064\97709031.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors="coerce")


In [27]:
print("=" * 100)
print("DATE LOGIC CHECKS")
print("=" * 100)

print(
    "\nTraining before StartDate:",
    (df["Training Date"] < df["StartDate"]).sum()
)

print(
    "Survey before StartDate:",
    (df["Survey Date"] < df["StartDate"]).sum()
)

print(
    "Training after SurveyDate:",
    (df["Training Date"] > df["Survey Date"]).sum()
)

print(
    "DOB after StartDate:",
    (df["DOB"] > df["StartDate"]).sum()
)

DATE LOGIC CHECKS

Training before StartDate: 273
Survey before StartDate: 272
Training after SurveyDate: 1403
DOB after StartDate: 0


In [28]:
df_raw = datasets["Cleaned_HR_Data_Analysis"].copy()

print("=" * 100)
print("RAW DOB EXAMPLES")
print("=" * 100)

print(df_raw["DOB"].head(30).to_string())

print("\n" + "=" * 100)
print("RAW DATE FORMAT EXAMPLES")
print("=" * 100)

for col in ["StartDate", "DOB", "Survey Date", "Training Date"]:
    print(f"\n{col}")
    print(df_raw[col].dropna().astype(str).head(20).tolist())

RAW DOB EXAMPLES
0     07-10-1969
1     30-08-1965
2     06-10-1991
3     04-04-1998
4     29-08-1969
5     03-04-1949
6     01-07-1942
7     07-03-1957
8     15-05-1974
9     11-11-1949
10    26-01-1964
11    06-04-1948
12    24-11-1981
13    06-11-1951
14    21-11-1989
15    24-11-1952
16    08-04-1994
17    15-11-1983
18    07-12-1985
19    01-05-1996
20    11-08-1994
21    15-01-1968
22    07-01-1947
23    07-04-1982
24    29-01-1970
25    18-01-1999
26    25-09-1946
27    23-08-1947
28    10-02-1944
29    09-08-1942

RAW DATE FORMAT EXAMPLES

StartDate
['20-Sep-19', '11-Feb-23', '10-Dec-18', '21-Jun-21', '29-Jun-19', '17-Jan-20', '06-Apr-22', '06-Nov-20', '18-Aug-18', '21-Jan-22', '04-Aug-23', '10-Aug-18', '25-May-22', '05-Dec-19', '28-Apr-19', '09-Jul-19', '05-Apr-21', '28-Nov-21', '16-Jan-21', '24-Aug-21']

DOB
['07-10-1969', '30-08-1965', '06-10-1991', '04-04-1998', '29-08-1969', '03-04-1949', '01-07-1942', '07-03-1957', '15-05-1974', '11-11-1949', '26-01-1964', '06-04-1948', '

In [29]:
print("=" * 100)
print("DOB VALUES THAT FAILED PREVIOUS PARSING")
print("=" * 100)

dob_test = pd.to_datetime(
    df_raw["DOB"],
    errors="coerce",
    dayfirst=True
)

failed = df_raw.loc[
    dob_test.isna(),
    ["Employee ID", "DOB", "StartDate", "Age"]
]

print("Failed DOB conversions:", len(failed))

display(failed.head(30))

DOB VALUES THAT FAILED PREVIOUS PARSING
Failed DOB conversions: 0


,Employee ID,DOB,StartDate,Age


In [30]:
df = datasets["Cleaned_HR_Data_Analysis"].copy()

# Parse dates using their actual formats
df["StartDate"] = pd.to_datetime(
    df["StartDate"],
    format="%d-%b-%y",
    errors="coerce"
)

df["DOB"] = pd.to_datetime(
    df["DOB"],
    format="%d-%m-%Y",
    errors="coerce"
)

df["Survey Date"] = pd.to_datetime(
    df["Survey Date"],
    format="%d-%m-%Y",
    errors="coerce"
)

df["Training Date"] = pd.to_datetime(
    df["Training Date"],
    format="%d-%b-%y",
    errors="coerce"
)

print("=" * 100)
print("DATE CONVERSION CHECK")
print("=" * 100)

for col in ["StartDate", "DOB", "Survey Date", "Training Date"]:
    print(f"{col}: {df[col].isna().sum()} failed conversions")

DATE CONVERSION CHECK
StartDate: 0 failed conversions
DOB: 0 failed conversions
Survey Date: 0 failed conversions
Training Date: 0 failed conversions


In [31]:
print("=" * 100)
print("DATE LOGIC CHECK AFTER CORRECT PARSING")
print("=" * 100)

print(
    "Training before StartDate:",
    (df["Training Date"] < df["StartDate"]).sum()
)

print(
    "Survey before StartDate:",
    (df["Survey Date"] < df["StartDate"]).sum()
)

print(
    "Training after SurveyDate:",
    (df["Training Date"] > df["Survey Date"]).sum()
)

print(
    "DOB after StartDate:",
    (df["DOB"] > df["StartDate"]).sum()
)

DATE LOGIC CHECK AFTER CORRECT PARSING
Training before StartDate: 273
Survey before StartDate: 272
Training after SurveyDate: 1403
DOB after StartDate: 0


In [32]:
df["CalculatedAgeAtJoining"] = (
    (df["StartDate"] - df["DOB"]).dt.days / 365.25
).round().astype(int)

comparison = df[
    df["Age"] != df["CalculatedAgeAtJoining"]
][
    ["Employee ID", "DOB", "StartDate", "Age", "CalculatedAgeAtJoining"]
]

print("=" * 100)
print("AGE CONSISTENCY CHECK")
print("=" * 100)

print("Age mismatches:", len(comparison))

display(comparison.head(20))

AGE CONSISTENCY CHECK
Age mismatches: 730


,Employee ID,DOB,StartDate,Age,CalculatedAgeAtJoining
1,3428,1965-08-30,2023-02-11,58,57
7,3434,1957-03-07,2020-11-06,63,64
9,3436,1949-11-11,2022-01-21,73,72
10,3437,1964-01-26,2023-08-04,59,60
12,3439,1981-11-24,2022-05-25,41,40
14,3441,1989-11-21,2019-04-28,30,29
18,3445,1985-12-07,2021-01-16,36,35
22,3452,1947-01-07,2022-11-08,75,76
23,3453,1982-04-07,2022-10-13,40,41
24,3454,1970-01-29,2022-09-11,52,53


In [33]:
# Calculate age at survey date
df["CalculatedAgeAtSurvey"] = (
    (df["Survey Date"] - df["DOB"]).dt.days / 365.25
).round().astype(int)

survey_age_mismatch = df[
    df["Age"] != df["CalculatedAgeAtSurvey"]
][
    ["Employee ID", "DOB", "Survey Date", "Age", "CalculatedAgeAtSurvey"]
]

print("=" * 100)
print("AGE VS SURVEY DATE")
print("=" * 100)

print("Mismatches:", len(survey_age_mismatch))
print("Total employees:", len(df))

display(survey_age_mismatch.head(20))

AGE VS SURVEY DATE
Mismatches: 2407
Total employees: 2845


,Employee ID,DOB,Survey Date,Age,CalculatedAgeAtSurvey
0,3427,1969-10-07,2023-01-14,50,53
1,3428,1965-08-30,2022-09-09,58,57
2,3429,1991-10-06,2023-05-27,27,32
3,3430,1998-04-04,2023-06-16,23,25
4,3431,1969-08-29,2022-11-25,50,53
5,3432,1949-04-03,2022-12-12,71,74
6,3433,1942-07-01,2023-03-25,80,81
7,3434,1957-03-07,2023-04-21,63,66
8,3435,1974-05-15,2022-12-09,44,49
9,3436,1949-11-11,2023-07-30,73,74


In [34]:
print("=" * 100)
print("AGE / START DATE ANALYSIS")
print("=" * 100)

# Age at joining
df["AgeAtJoining"] = (
    (df["StartDate"] - df["DOB"]).dt.days / 365.25
).round().astype(int)

df["AgeDifference"] = df["Age"] - df["AgeAtJoining"]

print("\nAge difference statistics:")
print(df["AgeDifference"].describe())

print("\nMost common age differences:")
print(df["AgeDifference"].value_counts().sort_index())

print("\n" + "=" * 100)
print("TRAINING BEFORE START DATE")
print("=" * 100)

training_before = df[df["Training Date"] < df["StartDate"]]

print("Records:", len(training_before))

print("\nBy Training Type:")
print(training_before["Training Type"].value_counts())

print("\nBy Training Program:")
print(training_before["Training Program Name"].value_counts())

print("\nBy Employee Status:")
print(training_before["EmployeeStatus"].value_counts())

AGE / START DATE ANALYSIS

Age difference statistics:
count    2845.000000
mean       -0.002812
std         0.506629
min        -1.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         1.000000
Name: AgeDifference, dtype: float64

Most common age differences:
AgeDifference
-1     369
 0    2115
 1     361
Name: count, dtype: int64

TRAINING BEFORE START DATE
Records: 273

By Training Type:
Training Type
External    148
Internal    125
Name: count, dtype: int64

By Training Program:
Training Program Name
Customer Service          59
Leadership Development    55
Communication Skills      54
Project Management        54
Technical Skills          51
Name: count, dtype: int64

By Employee Status:
EmployeeStatus
Active        242
Terminated     31
Name: count, dtype: int64


In [35]:
print("=" * 100)
print("SURVEY BEFORE START DATE ANALYSIS")
print("=" * 100)

survey_before = df[df["Survey Date"] < df["StartDate"]].copy()

print("\nRecords:", len(survey_before))
print("Percentage:", round(len(survey_before) / len(df) * 100, 2), "%")

print("\nBy Employee Status:")
print(survey_before["EmployeeStatus"].value_counts())

print("\nBy Employee Type:")
print(survey_before["EmployeeType"].value_counts())

print("\nBy Department:")
print(survey_before["DepartmentType"].value_counts())

print("\nBy Training Type:")
print(survey_before["Training Type"].value_counts())

print("\nBy Training Program:")
print(survey_before["Training Program Name"].value_counts())

print("\nSample records:")
print(
    survey_before[
        [
            "Employee ID",
            "StartDate",
            "Survey Date",
            "Training Date",
            "EmployeeStatus",
            "EmployeeType",
            "DepartmentType",
            "Title"
        ]
    ].head(20).to_string(index=False)
)

SURVEY BEFORE START DATE ANALYSIS

Records: 272
Percentage: 9.56 %

By Employee Status:
EmployeeStatus
Active        247
Terminated     25
Name: count, dtype: int64

By Employee Type:
EmployeeType
Contract     95
Part-Time    91
Full-Time    86
Name: count, dtype: int64

By Department:
DepartmentType
Production              188
IT/IS                    31
Sales                    30
Software Engineering     11
Admin Offices            10
Executive Office          2
Name: count, dtype: int64

By Training Type:
Training Type
External    138
Internal    134
Name: count, dtype: int64

By Training Program:
Training Program Name
Communication Skills      65
Customer Service          56
Leadership Development    52
Project Management        50
Technical Skills          49
Name: count, dtype: int64

Sample records:
 Employee ID  StartDate Survey Date Training Date EmployeeStatus EmployeeType    DepartmentType                    Title
        3428 2023-02-11  2022-09-09    2022-09-12         Ac

In [36]:
# ============================================================
# STEP 2A — DATASET STRUCTURE MAPPING
# ============================================================

print("=" * 100)
print("DATASET STRUCTURE & KEY COLUMN ANALYSIS")
print("=" * 100)

for name, df in datasets.items():

    print("\n" + "=" * 100)
    print(f"DATASET: {name}")
    print("=" * 100)

    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")

    print("\nColumns:")
    for col in df.columns:
        print(f"  - {col}")

    print("\nPotential ID columns:")
    id_cols = [
        col for col in df.columns
        if any(keyword in col.lower()
               for keyword in ["id", "employee", "number", "code"])
    ]

    if id_cols:
        for col in id_cols:
            print(
                f"  {col} → "
                f"unique={df[col].nunique()}, "
                f"missing={df[col].isna().sum()}"
            )
    else:
        print("  None detected")

    print("\nPotential job/occupation columns:")
    job_cols = [
        col for col in df.columns
        if any(keyword in col.lower()
               for keyword in ["title", "role", "occupation", "job"])
    ]

    print(job_cols if job_cols else "None")

    print("\nPotential performance columns:")
    perf_cols = [
        col for col in df.columns
        if any(keyword in col.lower()
               for keyword in ["performance", "rating", "kpi", "feedback"])
    ]

    print(perf_cols if perf_cols else "None")

    print("\nPotential engagement/satisfaction columns:")
    engagement_cols = [
        col for col in df.columns
        if any(keyword in col.lower()
               for keyword in [
                   "engagement",
                   "satisfaction",
                   "work-life",
                   "worklife"
               ])
    ]

    print(engagement_cols if engagement_cols else "None")

    print("\nPotential training columns:")
    training_cols = [
        col for col in df.columns
        if "training" in col.lower()
    ]

    print(training_cols if training_cols else "None")

    print("\nPotential attrition/employment-status columns:")
    attrition_cols = [
        col for col in df.columns
        if any(keyword in col.lower()
               for keyword in [
                   "attrition",
                   "status",
                   "termination",
                   "exit"
               ])
    ]

    print(attrition_cols if attrition_cols else "None")

DATASET STRUCTURE & KEY COLUMN ANALYSIS

DATASET: Cleaned_HR_Data_Analysis
Rows: 2845
Columns: 28

Columns:
  - Employee ID
  - StartDate
  - Title
  - BusinessUnit
  - EmployeeStatus
  - EmployeeType
  - PayZone
  - EmployeeClassificationType
  - DepartmentType
  - Division
  - DOB
  - State
  - GenderCode
  - RaceDesc
  - MaritalDesc
  - Performance Score
  - Current Employee Rating
  - Survey Date
  - Engagement Score
  - Satisfaction Score
  - Work-Life Balance Score
  - Training Date
  - Training Program Name
  - Training Type
  - Training Outcome
  - Training Duration(Days)
  - Training Cost
  - Age

Potential ID columns:
  Employee ID → unique=2845, missing=0
  EmployeeStatus → unique=2, missing=0
  EmployeeType → unique=3, missing=0
  EmployeeClassificationType → unique=3, missing=0
  GenderCode → unique=2, missing=0
  Current Employee Rating → unique=5, missing=0

Potential job/occupation columns:
['Title']

Potential performance columns:
['Performance Score', 'Current Employe

In [37]:
# ============================================================
# STEP 2B — EMPLOYEE ID RELATIONSHIP ANALYSIS
# ============================================================

hr_clean = datasets["Cleaned_HR_Data_Analysis"]
hr_messy = datasets["Messy_HR_Dataset_Detailed"]

clean_ids = set(hr_clean["Employee ID"])
messy_ids = set(hr_messy["Employee ID"])

print("=" * 100)
print("EMPLOYEE ID RELATIONSHIP")
print("=" * 100)

print("\nClean HR unique IDs:", len(clean_ids))
print("Messy HR unique IDs:", len(messy_ids))

print("\nIDs common to both:", len(clean_ids & messy_ids))

print(
    "Clean HR IDs missing from Messy:",
    len(clean_ids - messy_ids)
)

print(
    "Messy HR IDs missing from Clean:",
    len(messy_ids - clean_ids)
)

print(
    "\nPercentage of Clean HR employees found in Messy:",
    round(len(clean_ids & messy_ids) / len(clean_ids) * 100, 2),
    "%"
)

print(
    "Percentage of Messy HR employees found in Clean:",
    round(len(clean_ids & messy_ids) / len(messy_ids) * 100, 2),
    "%"
)

print("\nSample IDs only in Clean:")
print(sorted(clean_ids - messy_ids)[:20])

print("\nSample IDs only in Messy:")
print(sorted(messy_ids - clean_ids)[:20])

EMPLOYEE ID RELATIONSHIP

Clean HR unique IDs: 2845
Messy HR unique IDs: 3000

IDs common to both: 2845
Clean HR IDs missing from Messy: 0
Messy HR IDs missing from Clean: 155

Percentage of Clean HR employees found in Messy: 100.0 %
Percentage of Messy HR employees found in Clean: 94.83 %

Sample IDs only in Clean:
[]

Sample IDs only in Messy:
[1195, 1348, 1354, 1361, 1422, 1424, 1445, 1446, 1447, 1488, 1511, 1512, 1531, 1532, 1533, 1551, 1552, 1554, 1574, 1575]


In [38]:
# ============================================================
# COMPARE COMMON EMPLOYEES: CLEAN HR vs MESSY HR
# ============================================================

clean = datasets["Cleaned_HR_Data_Analysis"].copy()
messy = datasets["Messy_HR_Dataset_Detailed"].copy()

# Remove exact duplicate records from messy dataset first
messy_unique = messy.drop_duplicates().copy()

# Keep only one record per Employee ID
messy_unique = messy_unique.drop_duplicates(subset=["Employee ID"], keep="first")

# Find common employees
common_ids = set(clean["Employee ID"]) & set(messy_unique["Employee ID"])

clean_common = clean[clean["Employee ID"].isin(common_ids)].copy()
messy_common = messy_unique[messy_unique["Employee ID"].isin(common_ids)].copy()

# Sort so employees line up
clean_common = clean_common.sort_values("Employee ID").reset_index(drop=True)
messy_common = messy_common.sort_values("Employee ID").reset_index(drop=True)

print("=" * 100)
print("COMMON EMPLOYEE COMPARISON")
print("=" * 100)

print("Common employees:", len(common_ids))
print("Clean rows:", len(clean_common))
print("Messy rows:", len(messy_common))

# Columns that exist in both datasets
common_columns = sorted(
    list(set(clean_common.columns) & set(messy_common.columns))
)

print("\nCommon columns:")
print(common_columns)

# Compare values
comparison_results = []

for col in common_columns:
    if col == "Employee ID":
        continue

    # Convert to string for safe comparison
    clean_values = clean_common[col].fillna("<NA>").astype(str).str.strip()
    messy_values = messy_common[col].fillna("<NA>").astype(str).str.strip()

    mismatches = (clean_values != messy_values).sum()

    comparison_results.append({
        "Column": col,
        "Mismatches": mismatches,
        "Match_Percentage": round(
            (1 - mismatches / len(common_ids)) * 100, 2
        )
    })

comparison_df = pd.DataFrame(comparison_results)

comparison_df = comparison_df.sort_values(
    "Mismatches",
    ascending=False
)

print("\n" + "=" * 100)
print("COLUMN-BY-COLUMN COMPARISON")
print("=" * 100)

display(comparison_df)

COMMON EMPLOYEE COMPARISON
Common employees: 2845
Clean rows: 2845
Messy rows: 2845

Common columns:
['BusinessUnit', 'Current Employee Rating', 'DOB', 'DepartmentType', 'Division', 'Employee ID', 'EmployeeClassificationType', 'EmployeeStatus', 'EmployeeType', 'Engagement Score', 'GenderCode', 'MaritalDesc', 'PayZone', 'Performance Score', 'RaceDesc', 'Satisfaction Score', 'StartDate', 'State', 'Survey Date', 'Title', 'Training Cost', 'Training Date', 'Training Duration(Days)', 'Training Outcome', 'Training Program Name', 'Training Type', 'Work-Life Balance Score']

COLUMN-BY-COLUMN COMPARISON


,Column,Mismatches,Match_Percentage
6,EmployeeStatus,387,86.4
0,BusinessUnit,0,100.0
1,Current Employee Rating,0,100.0
2,DOB,0,100.0
4,Division,0,100.0
3,DepartmentType,0,100.0
5,EmployeeClassificationType,0,100.0
7,EmployeeType,0,100.0
8,Engagement Score,0,100.0
9,GenderCode,0,100.0


In [40]:
# ============================================================
# EMPLOYEE STATUS MISMATCH ANALYSIS
# ============================================================

clean = datasets["Cleaned_HR_Data_Analysis"].copy()
messy = datasets["Messy_HR_Dataset_Detailed"].copy()

# Remove exact duplicate rows
messy_unique = messy.drop_duplicates()

# Keep one record per employee
messy_unique = messy_unique.drop_duplicates(
    subset=["Employee ID"],
    keep="first"
)

# Merge status information
status_check = clean[["Employee ID", "EmployeeStatus"]].merge(
    messy_unique[["Employee ID", "EmployeeStatus"]],
    on="Employee ID",
    suffixes=("_Clean", "_Messy")
)

# Keep only mismatches
status_mismatches = status_check[
    status_check["EmployeeStatus_Clean"] !=
    status_check["EmployeeStatus_Messy"]
].copy()

print("=" * 100)
print("EMPLOYEE STATUS MISMATCH ANALYSIS")
print("=" * 100)

print("Total mismatched employees:", len(status_mismatches))

print("\nStatus mapping:")
display(
    pd.crosstab(
        status_mismatches["EmployeeStatus_Clean"],
        status_mismatches["EmployeeStatus_Messy"],
        margins=True
    )
)

print("\nSample mismatched records:")
display(status_mismatches.head(20))

EMPLOYEE STATUS MISMATCH ANALYSIS
Total mismatched employees: 387

Status mapping:


EmployeeStatus_Messy,Terminated for Cause,Voluntarily Terminated,All
EmployeeStatus_Clean,,,
Terminated,66,321,387
All,66,321,387



Sample mismatched records:


,Employee ID,EmployeeStatus_Clean,EmployeeStatus_Messy
116,3564,Terminated,Voluntarily Terminated
120,3568,Terminated,Voluntarily Terminated
122,3570,Terminated,Voluntarily Terminated
123,3571,Terminated,Voluntarily Terminated
126,3575,Terminated,Voluntarily Terminated
145,3598,Terminated,Voluntarily Terminated
149,3602,Terminated,Voluntarily Terminated
150,3603,Terminated,Voluntarily Terminated
153,3606,Terminated,Voluntarily Terminated
171,3625,Terminated,Voluntarily Terminated


In [41]:
# ============================================================
# ANALYZE MESSY-ONLY EMPLOYEES
# ============================================================

clean = datasets["Cleaned_HR_Data_Analysis"].copy()
messy = datasets["Messy_HR_Dataset_Detailed"].copy()

# Remove exact duplicates and keep one record per employee
messy_unique = messy.drop_duplicates()
messy_unique = messy_unique.drop_duplicates(
    subset=["Employee ID"],
    keep="first"
)

clean_ids = set(clean["Employee ID"])
messy_ids = set(messy_unique["Employee ID"])

messy_only_ids = messy_ids - clean_ids

messy_only = messy_unique[
    messy_unique["Employee ID"].isin(messy_only_ids)
].copy()

print("=" * 100)
print("MESSY-ONLY EMPLOYEE ANALYSIS")
print("=" * 100)

print("Messy-only employees:", len(messy_only))

print("\nEmployee Status:")
display(messy_only["EmployeeStatus"].value_counts())

print("\nTermination Type:")
display(messy_only["TerminationType"].value_counts())

print("\nEmployee Type:")
display(messy_only["EmployeeType"].value_counts())

print("\nDepartment:")
display(messy_only["DepartmentType"].value_counts())

print("\nSample messy-only employees:")
display(
    messy_only[
        [
            "Employee ID",
            "FirstName",
            "LastName",
            "StartDate",
            "ExitDate",
            "Title",
            "EmployeeStatus",
            "EmployeeType",
            "DepartmentType",
            "TerminationType"
        ]
    ].head(20)
)

MESSY-ONLY EMPLOYEE ANALYSIS
Messy-only employees: 155

Employee Status:


EmployeeStatus
Leave of Absence    86
Future Start        69
Name: count, dtype: int64


Termination Type:


TerminationType
Retirement     47
Voluntary      39
Involuntary    37
Resignation    32
Name: count, dtype: int64


Employee Type:


EmployeeType
Part-Time    57
Contract     57
Full-Time    41
Name: count, dtype: int64


Department:


DepartmentType
Production              110
IT/IS                    21
Sales                    20
Software Engineering      3
Admin Offices             1
Name: count, dtype: int64


Sample messy-only employees:


,Employee ID,FirstName,LastName,StartDate,ExitDate,Title,EmployeeStatus,EmployeeType,DepartmentType,TerminationType
20,3447,Mariela,Schultz,26-May-20,18-Jun-23,Area Sales Manager,Future Start,Part-Time,Sales,Involuntary
21,3448,Angela,Molina,01-Oct-19,06-Nov-20,Area Sales Manager,Future Start,Full-Time,Sales,Retirement
22,3449,Gerald,Preston,10-May-23,27-May-23,Area Sales Manager,Future Start,Contract,Sales,Involuntary
31,3458,Cory,Robinson,28-Apr-22,24-May-23,Area Sales Manager,Future Start,Contract,Sales,Voluntary
32,3459,Saniya,Yu,18-Apr-21,21-Jun-22,Area Sales Manager,Future Start,Part-Time,Sales,Retirement
34,3461,Lincoln,Compton,18-Jul-19,01-Oct-21,Area Sales Manager,Future Start,Full-Time,Sales,Resignation
42,3469,Ryland,Shepherd,29-Jul-20,05-Jan-23,Area Sales Manager,Future Start,Contract,Sales,Resignation
43,3470,Esteban,Gilbert,14-Nov-18,28-Oct-19,Area Sales Manager,Future Start,Contract,Sales,Retirement
53,3480,Jerimiah,Harmon,08-Sep-22,16-Oct-22,Area Sales Manager,Future Start,Full-Time,Sales,Retirement
54,3481,Leland,Allen,07-Mar-19,31-May-21,Area Sales Manager,Future Start,Full-Time,Sales,Voluntary


In [42]:
# ============================================================
# VALIDATE THE 155 MESSY-ONLY EMPLOYEES
# ============================================================

messy_only["StartDate"] = pd.to_datetime(
    messy_only["StartDate"],
    format="%d-%b-%y",
    errors="coerce"
)

messy_only["ExitDate"] = pd.to_datetime(
    messy_only["ExitDate"],
    format="%d-%b-%y",
    errors="coerce"
)

messy_only["DOB"] = pd.to_datetime(
    messy_only["DOB"],
    format="%d-%m-%Y",
    errors="coerce"
)

messy_only["Survey Date"] = pd.to_datetime(
    messy_only["Survey Date"],
    format="%d-%m-%Y",
    errors="coerce"
)

messy_only["Training Date"] = pd.to_datetime(
    messy_only["Training Date"],
    format="%d-%b-%y",
    errors="coerce"
)

print("=" * 100)
print("155 MESSY-ONLY EMPLOYEES: DATA QUALITY ANALYSIS")
print("=" * 100)

# ------------------------------------------------------------
# 1. ExitDate vs StartDate
# ------------------------------------------------------------

exit_before_start = (
    messy_only["ExitDate"].notna() &
    messy_only["StartDate"].notna() &
    (messy_only["ExitDate"] < messy_only["StartDate"])
).sum()

exit_present = messy_only["ExitDate"].notna().sum()

print("\nEXIT DATE")
print("-" * 60)
print("Exit dates present:", exit_present)
print("Exit dates before StartDate:", exit_before_start)

# ------------------------------------------------------------
# 2. Future Start + ExitDate
# ------------------------------------------------------------

future_start_exit = (
    (messy_only["EmployeeStatus"] == "Future Start") &
    messy_only["ExitDate"].notna()
).sum()

print("\nFUTURE START + EXIT DATE")
print("-" * 60)
print("Future Start employees:", 
      (messy_only["EmployeeStatus"] == "Future Start").sum())

print("Future Start employees with ExitDate:", future_start_exit)

# ------------------------------------------------------------
# 3. Leave of Absence + ExitDate
# ------------------------------------------------------------

loa_exit = (
    (messy_only["EmployeeStatus"] == "Leave of Absence") &
    messy_only["ExitDate"].notna()
).sum()

print("\nLEAVE OF ABSENCE + EXIT DATE")
print("-" * 60)
print("Leave of Absence employees:",
      (messy_only["EmployeeStatus"] == "Leave of Absence").sum())

print("Leave of Absence with ExitDate:", loa_exit)

# ------------------------------------------------------------
# 4. Survey before StartDate
# ------------------------------------------------------------

survey_before_start = (
    messy_only["Survey Date"].notna() &
    messy_only["StartDate"].notna() &
    (messy_only["Survey Date"] < messy_only["StartDate"])
).sum()

print("\nSURVEY BEFORE START DATE")
print("-" * 60)
print("Records:", survey_before_start)

# ------------------------------------------------------------
# 5. Training before StartDate
# ------------------------------------------------------------

training_before_start = (
    messy_only["Training Date"].notna() &
    messy_only["StartDate"].notna() &
    (messy_only["Training Date"] < messy_only["StartDate"])
).sum()

print("\nTRAINING BEFORE START DATE")
print("-" * 60)
print("Records:", training_before_start)

# ------------------------------------------------------------
# 6. Show suspicious Future Start records
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("FUTURE START EMPLOYEES WITH EXIT DATES")
print("=" * 100)

future_start_suspicious = messy_only[
    (messy_only["EmployeeStatus"] == "Future Start") &
    messy_only["ExitDate"].notna()
]

display(
    future_start_suspicious[
        [
            "Employee ID",
            "StartDate",
            "ExitDate",
            "Title",
            "EmployeeStatus",
            "EmployeeType",
            "DepartmentType",
            "TerminationType"
        ]
    ].head(30)
)

155 MESSY-ONLY EMPLOYEES: DATA QUALITY ANALYSIS

EXIT DATE
------------------------------------------------------------
Exit dates present: 155
Exit dates before StartDate: 0

FUTURE START + EXIT DATE
------------------------------------------------------------
Future Start employees: 69
Future Start employees with ExitDate: 69

LEAVE OF ABSENCE + EXIT DATE
------------------------------------------------------------
Leave of Absence employees: 86
Leave of Absence with ExitDate: 86

SURVEY BEFORE START DATE
------------------------------------------------------------
Records: 15

TRAINING BEFORE START DATE
------------------------------------------------------------
Records: 15

FUTURE START EMPLOYEES WITH EXIT DATES


,Employee ID,StartDate,ExitDate,Title,EmployeeStatus,EmployeeType,DepartmentType,TerminationType
20,3447,2020-05-26,2023-06-18,Area Sales Manager,Future Start,Part-Time,Sales,Involuntary
21,3448,2019-10-01,2020-11-06,Area Sales Manager,Future Start,Full-Time,Sales,Retirement
22,3449,2023-05-10,2023-05-27,Area Sales Manager,Future Start,Contract,Sales,Involuntary
31,3458,2022-04-28,2023-05-24,Area Sales Manager,Future Start,Contract,Sales,Voluntary
32,3459,2021-04-18,2022-06-21,Area Sales Manager,Future Start,Part-Time,Sales,Retirement
34,3461,2019-07-18,2021-10-01,Area Sales Manager,Future Start,Full-Time,Sales,Resignation
42,3469,2020-07-29,2023-01-05,Area Sales Manager,Future Start,Contract,Sales,Resignation
43,3470,2018-11-14,2019-10-28,Area Sales Manager,Future Start,Contract,Sales,Retirement
53,3480,2022-09-08,2022-10-16,Area Sales Manager,Future Start,Full-Time,Sales,Retirement
54,3481,2019-03-07,2021-05-31,Area Sales Manager,Future Start,Full-Time,Sales,Voluntary


In [44]:
print("df shape:", df.shape)
print("df columns:")
print(df.columns.tolist())

print("\n" + "="*80)

print("messy shape:", messy.shape)
print("messy columns:")
print(messy.columns.tolist())

df shape: (1470, 35)
df columns:
['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department', 'DistanceFromHome', 'Education', 'EducationField', 'EmployeeCount', 'EmployeeNumber', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'Over18', 'OverTime', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StandardHours', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']

messy shape: (3150, 39)
messy columns:
['Unnamed: 0', 'FirstName', 'LastName', 'StartDate', 'ExitDate', 'Title', 'Supervisor', 'ADEmail', 'BusinessUnit', 'EmployeeStatus', 'EmployeeType', 'PayZone', 'EmployeeClassificationType', 'TerminationType', 'TerminationDescription', 'DepartmentType', 'Division', 'DOB', 'State', 'JobFunctionDescription', '

In [46]:
import pandas as pd

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        print(f"{name:<30} {obj.shape}")

_                              (8, 6)
df                             (1470, 35)
audit_df                       (8, 6)
_4                             (8, 6)
missing_df                     (2, 2)
messy                          (3150, 39)
duplicate_sample               (10, 39)
active_with_exit               (1039, 39)
active_with_termination        (1039, 39)
inactive_without_exit          (0, 39)
status_termination             (6, 6)
exit_analysis                  (5, 4)
termination_analysis           (5, 3)
onet                           (1016, 3)
df_raw                         (2845, 28)
failed                         (0, 4)
comparison                     (730, 5)
survey_age_mismatch            (2407, 5)
training_before                (273, 32)
survey_before                  (272, 32)
hr_clean                       (2845, 28)
hr_messy                       (3150, 39)
clean                          (2845, 28)
messy_unique                   (3000, 39)
clean_common                   (284

In [47]:
# ============================================================
# LOCK DATASET ROLES
# ============================================================

# Primary HR dataset
clean_hr = clean.copy()

# Original messy HR dataset
messy_hr = messy.copy()

# External attrition dataset
attrition_hr = df.copy()

# O*NET occupation dataset
onet_hr = onet.copy()


# ============================================================
# VERIFY
# ============================================================

print("=" * 80)
print("DATASET ROLES")
print("=" * 80)

print(f"clean_hr      : {clean_hr.shape}")
print(f"messy_hr      : {messy_hr.shape}")
print(f"attrition_hr  : {attrition_hr.shape}")
print(f"onet_hr       : {onet_hr.shape}")

DATASET ROLES
clean_hr      : (2845, 28)
messy_hr      : (3150, 39)
attrition_hr  : (1470, 35)
onet_hr       : (1016, 3)


In [49]:
# ============================================================
# FINAL DATASET ROLE LOCK
# ============================================================

# Our canonical datasets
clean_hr = hr_clean.copy()
messy_hr = hr_messy.copy()
attrition_hr = df.copy()
onet_hr = onet.copy()

print("=" * 80)
print("DATASET ROLES LOCKED")
print("=" * 80)

print(f"clean_hr      : {clean_hr.shape}")
print(f"messy_hr      : {messy_hr.shape}")
print(f"attrition_hr  : {attrition_hr.shape}")
print(f"onet_hr       : {onet_hr.shape}")

print("\nClean HR columns:")
print(clean_hr.columns.tolist())

print("\nMessy HR columns:")
print(messy_hr.columns.tolist())

DATASET ROLES LOCKED
clean_hr      : (2845, 28)
messy_hr      : (3150, 39)
attrition_hr  : (1470, 35)
onet_hr       : (1016, 3)

Clean HR columns:
['Employee ID', 'StartDate', 'Title', 'BusinessUnit', 'EmployeeStatus', 'EmployeeType', 'PayZone', 'EmployeeClassificationType', 'DepartmentType', 'Division', 'DOB', 'State', 'GenderCode', 'RaceDesc', 'MaritalDesc', 'Performance Score', 'Current Employee Rating', 'Survey Date', 'Engagement Score', 'Satisfaction Score', 'Work-Life Balance Score', 'Training Date', 'Training Program Name', 'Training Type', 'Training Outcome', 'Training Duration(Days)', 'Training Cost', 'Age']

Messy HR columns:
['Unnamed: 0', 'FirstName', 'LastName', 'StartDate', 'ExitDate', 'Title', 'Supervisor', 'ADEmail', 'BusinessUnit', 'EmployeeStatus', 'EmployeeType', 'PayZone', 'EmployeeClassificationType', 'TerminationType', 'TerminationDescription', 'DepartmentType', 'Division', 'DOB', 'State', 'JobFunctionDescription', 'GenderCode', 'LocationCode', 'RaceDesc', 'Marita

In [50]:
# ============================================================
# DATE AUDIT
# ============================================================

audit_dates = clean_hr.copy()

audit_dates["StartDate"] = pd.to_datetime(
    audit_dates["StartDate"],
    format="%d-%b-%y",
    errors="coerce"
)

audit_dates["DOB"] = pd.to_datetime(
    audit_dates["DOB"],
    format="%d-%m-%Y",
    errors="coerce"
)

audit_dates["Survey Date"] = pd.to_datetime(
    audit_dates["Survey Date"],
    format="%d-%m-%Y",
    errors="coerce"
)

audit_dates["Training Date"] = pd.to_datetime(
    audit_dates["Training Date"],
    format="%d-%b-%y",
    errors="coerce"
)

print("=" * 80)
print("DATE CONVERSION AUDIT")
print("=" * 80)

for col in ["StartDate", "DOB", "Survey Date", "Training Date"]:
    print(
        f"{col:20} failed conversions: "
        f"{audit_dates[col].isna().sum()}"
    )

DATE CONVERSION AUDIT
StartDate            failed conversions: 0
DOB                  failed conversions: 0
Survey Date          failed conversions: 0
Training Date        failed conversions: 0


In [51]:
# ============================================================
# DATE LOGIC AUDIT
# ============================================================

start = audit_dates["StartDate"]
dob = audit_dates["DOB"]
survey = audit_dates["Survey Date"]
training = audit_dates["Training Date"]

print("=" * 80)
print("DATE LOGIC AUDIT")
print("=" * 80)

print(
    "Training before StartDate :",
    (training < start).sum()
)

print(
    "Survey before StartDate   :",
    (survey < start).sum()
)

print(
    "DOB after StartDate       :",
    (dob > start).sum()
)

DATE LOGIC AUDIT
Training before StartDate : 273
Survey before StartDate   : 272
DOB after StartDate       : 0


In [52]:
# ============================================================
# AGE CONSISTENCY AUDIT
# ============================================================

def calculate_age(birth_date, reference_date):
    return (
        reference_date.dt.year
        - birth_date.dt.year
        - (
            (reference_date.dt.month < birth_date.dt.month)
            |
            (
                (reference_date.dt.month == birth_date.dt.month)
                &
                (reference_date.dt.day < birth_date.dt.day)
            )
        )
    )


audit_dates["CalculatedAgeAtJoining"] = calculate_age(
    audit_dates["DOB"],
    audit_dates["StartDate"]
)

audit_dates["AgeDifference"] = (
    clean_hr["Age"]
    - audit_dates["CalculatedAgeAtJoining"]
)

print("=" * 80)
print("AGE CONSISTENCY AUDIT")
print("=" * 80)

print("\nAge difference distribution:")
print(
    audit_dates["AgeDifference"]
    .value_counts()
    .sort_index()
)

print("\nAge difference statistics:")
print(
    audit_dates["AgeDifference"].describe()
)

AGE CONSISTENCY AUDIT

Age difference distribution:
AgeDifference
0    1463
1    1382
Name: count, dtype: int64

Age difference statistics:
count    2845.000000
mean        0.485764
std         0.499885
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max         1.000000
Name: AgeDifference, dtype: float64


In [54]:
# ============================================================
# 5. AGE CONSISTENCY
# ============================================================

print("\n5. AGE CONSISTENCY")
print("-" * 100)

# Make sure dates are datetime
clean_hr["StartDate"] = pd.to_datetime(
    clean_hr["StartDate"],
    format="%d-%b-%y",
    errors="coerce"
)

clean_hr["DOB"] = pd.to_datetime(
    clean_hr["DOB"],
    format="%d-%m-%Y",
    errors="coerce"
)

# Calculate actual age at joining
calculated_age = (
    clean_hr["StartDate"] - clean_hr["DOB"]
).dt.days / 365.25

calculated_age = calculated_age.astype(int)

# Difference between dataset Age and calculated age
age_difference = clean_hr["Age"] - calculated_age

# Store temporarily in the dataframe
clean_hr["AgeDifference"] = age_difference

# Distribution
age_counts = clean_hr["AgeDifference"].value_counts().sort_index()

print("\nAge difference distribution:")

for diff, count in age_counts.items():
    percentage = count / len(clean_hr) * 100
    print(f"Difference {diff:+}: {count} employees ({percentage:.2f}%)")

# Statistics
print("\nAge difference statistics:")
print(clean_hr["AgeDifference"].describe())

# Mismatches
age_mismatches = clean_hr[
    clean_hr["AgeDifference"] != 0
]

print("\nAge mismatches:")
print(f"Count: {len(age_mismatches)}")
print(f"Percentage: {len(age_mismatches) / len(clean_hr) * 100:.2f}%")

# Sample
if len(age_mismatches) > 0:
    print("\nSample age mismatches:")
    print(
        age_mismatches[
            [
                "Employee ID",
                "DOB",
                "StartDate",
                "Age",
                "AgeDifference"
            ]
        ].head(20).to_string(index=False)
    )


5. AGE CONSISTENCY
----------------------------------------------------------------------------------------------------

Age difference distribution:
Difference +0: 1458 employees (51.25%)
Difference +1: 1387 employees (48.75%)

Age difference statistics:
count    2845.000000
mean        0.487522
std         0.499932
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max         1.000000
Name: AgeDifference, dtype: float64

Age mismatches:
Count: 1387
Percentage: 48.75%

Sample age mismatches:
 Employee ID        DOB  StartDate  Age  AgeDifference
        3427 1969-10-07 2019-09-20   50              1
        3428 1965-08-30 2023-02-11   58              1
        3431 1969-08-29 2019-06-29   50              1
        3432 1949-04-03 2020-01-17   71              1
        3433 1942-07-01 2022-04-06   80              1
        3436 1949-11-11 2022-01-21   73              1
        3439 1981-11-24 2022-05-25   41              1
        3441 1989-11-21 201

In [56]:
# ============================================================
# AGE AUDIT DATAFRAME
# ============================================================

age_audit = clean_hr[
    ["Employee ID", "DOB", "StartDate", "Age"]
].copy()

# Ensure dates are correctly parsed
age_audit["DOB"] = pd.to_datetime(
    age_audit["DOB"],
    format="%d-%m-%Y",
    errors="coerce"
)

age_audit["StartDate"] = pd.to_datetime(
    age_audit["StartDate"],
    format="%d-%b-%y",
    errors="coerce"
)

# Calculate age at joining
age_audit["CalculatedAgeAtJoining"] = (
    (age_audit["StartDate"] - age_audit["DOB"]).dt.days / 365.25
).astype(int)

# Difference between source Age and calculated age
age_audit["AgeDifference"] = (
    age_audit["Age"] - age_audit["CalculatedAgeAtJoining"]
)

print("AGE AUDIT CREATED")
print("=" * 80)
print(age_audit["AgeDifference"].value_counts().sort_index())

AGE AUDIT CREATED
AgeDifference
0    1458
1    1387
Name: count, dtype: int64


In [57]:
# ============================================================
# FINAL DATA QUALITY SUMMARY
# ============================================================

print("=" * 100)
print("FINAL DATA QUALITY SUMMARY")
print("=" * 100)

print("\nDATASET SIZE")
print("-" * 100)
print(f"Clean HR rows              : {len(clean_hr):,}")
print(f"Clean HR unique employees  : {clean_hr['Employee ID'].nunique():,}")
print(f"Messy HR rows              : {len(messy_hr):,}")
print(f"Messy HR unique employees  : {messy_hr['Employee ID'].nunique():,}")

print("\nEMPLOYEE ID INTEGRITY")
print("-" * 100)

clean_ids = set(clean_hr["Employee ID"])
messy_ids = set(messy_hr["Employee ID"])

print(f"Common IDs                 : {len(clean_ids & messy_ids):,}")
print(f"Clean-only IDs             : {len(clean_ids - messy_ids):,}")
print(f"Messy-only IDs             : {len(messy_ids - clean_ids):,}")

print("\nDUPLICATES")
print("-" * 100)
print(
    f"Clean duplicate IDs       : "
    f"{clean_hr['Employee ID'].duplicated().sum():,}"
)

print(
    f"Messy duplicate IDs       : "
    f"{messy_hr['Employee ID'].duplicated().sum():,}"
)

print("\nMISSING VALUES")
print("-" * 100)
print(
    f"Clean HR missing cells    : "
    f"{clean_hr.isna().sum().sum():,}"
)

print(
    f"Messy HR missing cells    : "
    f"{messy_hr.isna().sum().sum():,}"
)

print("\nDATE LOGIC")
print("-" * 100)

print(f"Training before StartDate : {len(training_before):,}")
print(f"Survey before StartDate   : {len(survey_before):,}")

print("\nAGE CONSISTENCY")
print("-" * 100)

age_counts = age_audit["AgeDifference"].value_counts().sort_index()

for diff, count in age_counts.items():
    percentage = count / len(age_audit) * 100
    print(
        f"Age difference {diff:+}   : "
        f"{count:,} ({percentage:.2f}%)"
    )

print("\nEMPLOYEE STATUS")
print("-" * 100)

print(
    f"Clean vs Messy status mismatches : "
    f"{len(status_mismatches):,}"
)

print(
    f"Messy-only employees             : "
    f"{len(messy_only):,}"
)

print("\nDECISIONS")
print("-" * 100)

print("1. Preserve source Age values.")
print("2. Do not treat Age +1 as random corruption.")
print("3. Preserve original Employee IDs.")
print("4. Do not blindly delete messy HR records.")
print("5. Investigate duplicate/multiple-status records separately.")
print("6. Date inconsistencies require business-rule decisions.")
print("7. Cleaning will be performed in the next notebook.")

FINAL DATA QUALITY SUMMARY

DATASET SIZE
----------------------------------------------------------------------------------------------------
Clean HR rows              : 2,845
Clean HR unique employees  : 2,845
Messy HR rows              : 3,150
Messy HR unique employees  : 3,000

EMPLOYEE ID INTEGRITY
----------------------------------------------------------------------------------------------------
Common IDs                 : 2,845
Clean-only IDs             : 0
Messy-only IDs             : 155

DUPLICATES
----------------------------------------------------------------------------------------------------
Clean duplicate IDs       : 0
Messy duplicate IDs       : 150

MISSING VALUES
----------------------------------------------------------------------------------------------------
Clean HR missing cells    : 0
Messy HR missing cells    : 3,088

DATE LOGIC
----------------------------------------------------------------------------------------------------
Training before StartDate 